In [1]:
import pandas as pd
import numpy as np

In [ ]:
class Commodity:
    
    #Risk weight table for Delta
    delta_risk_weights = {1: 0.3, 2: 0.35, 3: 0.6, 4: 0.8, 5: 0.4, 6: 0.45, 7: 0.2, 8: 0.35, 9: 0.25, 10: 0.35, 11: 0.5}
    
    #correlation for commodity MAR21.83(1)
    intra_corr_cty = {1: 0.55, 2: 0.95, 3: 0.4, 4: 0.8, 5: 0.6, 6: 0.65, 7: 0.55, 8: 0.45, 9: 0.15, 10: 0.4, 11: 0.15}
    
    def __init__(self, data):
        #Assume the data has already been cleansed, filtered and grouped
        self.original_data = data 
        self.data = data.copy()
        
    def weight_sensitivities(self):
        self.data["Weighted Sensitivities"] = self.data["Sensitivity (reporting currency equiv.)"] * np.select([self.data["Sensi Type"] == "Delta"], [self.data["SA Bucket"].map(Commodity.delta_risk_weights)], default=1)
        
    def perform_intra_bucket_aggregation(self, sensi_type):
        #Filter data to only include targeted sensitivity type
        filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
        
        #Aggregate data of the same sensitivities
        if sensi_type != "Curvature":
            cols_to_grp = ["SA Bucket", "Commodity", "Tenor", "Location"]
        else:
            cols_to_grp = ["SA Bucket", "Commodity", "CVR+/CVR-"]
        
        filtered_data = filtered_data.groupby(cols_to_grp)["Weighted Sensitivities"].sum().reset_index()
        
        #Pivot in case of CVR to have separate columns CVR+ and CVR-
        if sensi_type == "Curvature":
            filtered_data = filtered_data.pivot_table(index=["SA Bucket", "Commodity"], columns="CVR+/CVR-", values="Weighted Sensitivities", aggfunc="sum").reset_index()
            
            #Initiate variables to store CVR and K for each bucket for curvature
            temp_CVR_plus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            temp_CVR_minus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            temp_Kb_minus_by_bucket = [[], [], []]   #[[medium], [high], [low]]
            
        #Initiate variables to store K for each bucket
        temp_Kb_by_bucket = [[], [], []]   #[[medium], [high], [low]]
        K_by_bucket = []
            
        #Create a list of unique buckets
        bucket_list = filtered_data["SA Bucket"].drop_duplicates()
        
        for bucket in bucket_list:
            #Filter data for each bucket
            bucket_data = filtered_data[filtered_data["SA Bucket"] == bucket].reset_index(drop=True)

            #Initiate variables to store correlation matrix for each bucket
            corr_matrix_by_bucket = dict.fromkeys(bucket_list, None)
            
            n = len(bucket_data)
            
            #Generate correlation matrix for each bucket
            if sensi_type != "Curvature":
                
                temp_corr_matrix = np.ones((n, n))
                
                for i in range(n):
                    for j in range(n):
                        if i != j:
                            
                            #Check if commodity is identical
                            if bucket_data.loc[i, "Commodity"] != bucket_data.loc[j, "Commodity"]:
                                temp_corr_matrix[i, j] = Commodity.intra_corr_cty.get(bucket)
                            
                            if sensi_type == "Delta":        
                                #Check if the tenor is the same
                                if bucket_data.loc[i, "Tenor"] != bucket_data.loc[j, "Tenor"]:
                                    temp_corr_matrix[i, j] *= 0.99
                                
                                #Check if the delivery location is the same
                                if bucket_data.loc[i, "Location"] != bucket_data.loc[j, "Location"]:
                                    temp_corr_matrix[i, j] *= 0.999

                            else: #Vega
                                
                                #Check option tenor
                                temp_corr_matrix[i, j] *= np.exp(
                                                            -0.01 * np.abs(bucket_data.loc[i, "Tenor"] - bucket_data.loc[j, "Tenor"]) / 
                                                            np.minimum(bucket_data.loc[i, "Tenor"], bucket_data.loc[j, "Tenor"]))
                                                
                                temp_corr_matrix[i, j] = np.minimum(temp_corr_matrix[i, j], 1)
                                    
                #High and low scenario
                temp_high_corr_matrix = np.minimum(1.25 * temp_corr_matrix, 1)
                temp_low_corr_matrix = np.maximum(2 * temp_corr_matrix - 1, 0.75 * temp_corr_matrix)
                
                #Store results
                corr_matrix_by_bucket[bucket] = [temp_corr_matrix, temp_high_corr_matrix, temp_low_corr_matrix]
                
                #Calculate sum of weighted sensitivities of the bucket
                weighted_sensi = np.asarray(bucket_data["Weighted Sensitivities"])
                
                #Aggregate
                for i in range(3):
                    temp_Kb_by_bucket[i].append(np.sqrt(np.maximum(0, np.dot(np.transpose(weighted_sensi),np.dot(weighted_sensi, corr_matrix_by_bucket[bucket][i])))))
            
            else:
                
                temp_corr_matrix = np.zeros((n, n))
                psi_matrix_CVR_plus = np.zeros((n, n))
                psi_matrix_CVR_minus = np.zeros((n, n))                
                
                for i in range(n):
                    for j in range(n):
                        if i != j:
                            #Check if commodity is identical to map correlation
                            if bucket_data.loc[i, "Commodity"] != bucket_data.loc[j, "Commodity"]:
                                temp_corr_matrix[i, j] = Commodity.intra_corr_cty.get(bucket) ** 2
                            
                            #Calculate psi matrix for CVR+ and CVR-
                            if bucket_data.loc[i, "CVR+"] >= 0 or bucket_data.loc[j, "CVR+"] >= 0:
                                psi_matrix_CVR_plus[i, j] = 1
                            
                            if bucket_data.loc[i, "CVR-"] >= 0 or bucket_data.loc[j, "CVR-"] >= 0:
                                psi_matrix_CVR_minus[i, j] = 1
                            
                #high and low scenario
                temp_high_corr_matrix = np.minimum(1.25 * temp_corr_matrix, 1)
                temp_low_corr_matrix = np.maximum(2 * temp_corr_matrix - 1, 0.75 * temp_corr_matrix)
                
                #Store results
                corr_matrix_by_bucket[bucket] = [temp_corr_matrix, temp_high_corr_matrix, temp_low_corr_matrix]

                #Calculate K for the bucket
                for i in range(3):
                    temp_CVR_plus_by_bucket[i].append(bucket_data["CVR+"].sum())
                    
                    temp_CVR_minus_by_bucket[i].append(bucket_data["CVR-"].sum())
                    
                    temp_Kb_by_bucket[i].append(np.sqrt(
                                                np.maximum(0, 
                                                np.square(np.maximum(0, bucket_data["CVR+"])).sum() + 
                                                np.dot(bucket_data["CVR+"], np.dot(np.transpose(bucket_data["CVR+"]), np.multiply(corr_matrix_by_bucket[bucket][i], psi_matrix_CVR_plus)))
                                                )))
                    
                    temp_Kb_minus_by_bucket[i].append(np.sqrt(
                                                np.maximum(0, 
                                                np.square(np.maximum(0, bucket_data["CVR-"])).sum() + 
                                                np.dot(bucket_data["CVR-"], np.dot(np.transpose(bucket_data["CVR-"]), np.multiply(corr_matrix_by_bucket[bucket][i], psi_matrix_CVR_minus)))
                                                )))
                
        if sensi_type != "Curvature":
            
            for i in range(3):
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list, "K": temp_Kb_by_bucket[i]}).reset_index(drop=True))
                
        else:
            
            for i in range(3):
                K_by_bucket.append(pd.DataFrame({"SA Bucket": bucket_list,
                                                "CVR+": temp_CVR_plus_by_bucket[i],
                                                "CVR-": temp_CVR_minus_by_bucket[i],
                                                "K+": temp_Kb_by_bucket[i], 
                                                "K-": temp_Kb_minus_by_bucket[i], 
                                                "K": np.maximum(temp_Kb_by_bucket[i], temp_Kb_minus_by_bucket[i])})
                                                .reset_index(drop=True))

        return K_by_bucket
    
    def perform_across_bucket_aggregation(self, sensi_type, intra_agg_result):
        
        capital_charge = []
        
        if sensi_type != "Curvature":
            
            filtered_data = self.data[self.data["Sensi Type"] == sensi_type]
            
            #Calculate Sb according to formulation in MAR21.4(5)
            Sb_by_bucket = filtered_data.groupby("SA Bucket")["Weighted Sensitivities"].sum().reset_index()
            
            #Generate the correlation matrix
            n = len(Sb_by_bucket)
            medium_corr_matrix = np.full((n, n), 0.2)
            for i in range(n):
                for j in range(n):
                    if i == j or (Sb_by_bucket.loc[i, "SA Bucket"] != 11 or Sb_by_bucket.loc[i, "SA Bucket"] != 11):
                        medium_corr_matrix[i, j] = 0
            
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
            
            #Calculate the capital charge under each scenario [medium, high, low]
            for i in range(3):
                Kb = intra_agg_result[i]["K"]
                Sb = Sb_by_bucket["Weighted Sensitivities"]
                capital_charge.append(np.sqrt(
                                        np.square(Kb).sum() + 
                                        np.dot(np.transpose(Sb), np.dot(Sb, corr_matrices[i]))
                                        ))
                
                #For each scenario, we have to check whether an alternative formulation for Sb is required as stipulated in MAR21.4(5)
                if capital_charge[i] < 0:
                    Sb_Alt = np.maximum(np.minimum(Sb, Kb), -1 * Kb)
                    capital_charge[i] = (np.sqrt(
                                            np.square(Kb).sum() + 
                                            np.dot(np.transpose(Sb_Alt),np.dot(Sb_Alt, corr_matrices[i]))
                                            ))
                    
        else:
            
            temp_K_by_bucket = intra_agg_result[0]
            n = len(temp_K_by_bucket)
            
            #Generate the correlation matrix
            medium_corr_matrix = np.full((n, n), 0.2 ** 2)
            for i in range(n):
                for j in range(n):
                    if i == j or (Sb_by_bucket.loc[i, "SA Bucket"] != 11 or Sb_by_bucket.loc[i, "SA Bucket"] != 11):
                        medium_corr_matrix[i, j] = 0
                        
            high_corr_matrix = np.minimum(1.25 * medium_corr_matrix, 1)
            low_corr_matrix = np.maximum(2 * medium_corr_matrix - 1, 0.75 * medium_corr_matrix)
            
            corr_matrices = [medium_corr_matrix, high_corr_matrix, low_corr_matrix]
            
            Sb = [[], [], []] #[[medium], [high], [low]]
            psi_matrix = [[], [], []] #[[medium], [high], [low]]
            
            for i in range(3):
                temp_K_by_bucket = intra_agg_result[i]
                
                for j in range(n):                
                    #Calculate Sb according to formulation in MAR21.5(4)
                    if temp_K_by_bucket["K"][j] == temp_K_by_bucket["K+"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR+"][j])
                    elif temp_K_by_bucket["K"][j] == temp_K_by_bucket["K-"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR-"][j])
                    elif temp_K_by_bucket["CVR+"][j] > temp_K_by_bucket["CVR-"][j]:
                        Sb[i].append(temp_K_by_bucket["CVR+"][j])
                    else:
                        Sb[i].append(temp_K_by_bucket["CVR-"][j])
            
                #Generate the psi matrix according to MAR21.5(4)(b)
                psi_matrix[i] = np.zeros((n, n))
                for k in range(n):
                    for l in range(n):
                        if Sb[i][k] >= 0 or Sb[i][l] >= 0:
                            psi_matrix[i][k, l] = 1
                
                #Store results                
                intra_agg_result[i]["Sb"] = Sb[i]
            
                #Calculate capital charge
                capital_charge.append(np.sqrt(
                                        np.maximum(0, 
                                        np.square(intra_agg_result[i]["K"]).sum() + 
                                        np.dot(Sb[i], np.dot(np.transpose(Sb[i]), np.multiply(corr_matrices[i], psi_matrix[i])))
                                        )))
        
        result = pd.DataFrame({"Medium": [capital_charge[0]], "High": [capital_charge[1]], "Low": [capital_charge[2]]}, index=[sensi_type])
        
        return result
        

In [ ]:
#For quick test
test = pd.read_excel("Data/temp_frtb_data.xlsx", "Commodity")

x = Commodity(test)
x.weight_sensitivities()
intra = x.perform_intra_bucket_aggregation("Curvature")
x.perform_across_bucket_aggregation("Curvature", intra)

,Medium,High,Low
CVR,16527.395248,12608.875128,19680.59591
